In [1]:
ls -lh rag.ipynb

-rw-r--r--@ 1 pragyansh  staff   337B May 27 16:33 rag.ipynb


In [2]:
# Cell 1 — Imports & sanity check
import os
from dotenv import load_dotenv
import sentence_transformers
import faiss
import langchain
import groq

# Load environment variables from .env
load_dotenv()

# Verify API key loaded
api_key = os.getenv("GROQ_API_KEY")
print(f"sentence-transformers: {sentence_transformers.__version__}")
print(f"faiss version:         {faiss.__version__}")
print(f"langchain:             {langchain.__version__}")
print(f"groq:                  {groq.__version__}")
print(f"API key loaded:        {'✓ Yes' if api_key else '✗ No — check .env file'}")

sentence-transformers: 5.5.1
faiss version:         1.14.2
langchain:             1.3.2
groq:                  1.2.0
API key loaded:        ✓ Yes


In [3]:
# Cell 2 — Prepare documents
documents = [
    {
        "id": "doc_1",
        "title": "What is the Hugging Face Hub?",
        "text": """The Hugging Face Hub is a platform with over 350,000 models, 75,000 datasets, 
        and 150,000 demo apps. It is the central place where the Hugging Face community shares 
        AI models, datasets, and demos. The Hub works as a central place where anyone can explore, 
        experiment, collaborate, and build technology with Machine Learning. Models on the Hub can 
        be used directly in your browser using the Inference API, or downloaded locally. 
        The Hub supports Git-based version control for all repositories."""
    },
    {
        "id": "doc_2", 
        "title": "What are Transformers?",
        "text": """Transformers are a type of neural network architecture that uses self-attention 
        mechanisms to process sequential data. Introduced in the paper 'Attention is All You Need' 
        in 2017, transformers have become the dominant architecture for NLP tasks. The key innovation 
        is the attention mechanism, which allows the model to weigh the importance of different words 
        when processing each word in a sequence. BERT uses a bidirectional transformer encoder, 
        while GPT uses a unidirectional transformer decoder. The HuggingFace transformers library 
        provides thousands of pretrained transformer models for NLP, vision, and audio tasks."""
    },
    {
        "id": "doc_3",
        "title": "What is Fine-tuning?",
        "text": """Fine-tuning is the process of taking a pretrained model and further training it 
        on a smaller, task-specific dataset. The pretrained model has already learned general 
        representations from large amounts of data. Fine-tuning adapts these representations to 
        a specific task. There are several approaches to fine-tuning: full fine-tuning updates all 
        model parameters, while parameter-efficient methods like LoRA (Low-Rank Adaptation) only 
        update a small subset of parameters. LoRA works by adding low-rank matrices to existing 
        weights, reducing trainable parameters by up to 99% while maintaining performance. 
        The HuggingFace PEFT library provides implementations of LoRA and other efficient methods."""
    },
    {
        "id": "doc_4",
        "title": "What are Datasets in HuggingFace?",
        "text": """The HuggingFace datasets library provides access to thousands of datasets for 
        machine learning tasks. Datasets are loaded using the load_dataset function, which 
        downloads and caches the data locally. Datasets support streaming for large datasets 
        that don't fit in memory. The library uses Apache Arrow as its backend for fast, 
        memory-efficient data processing. Datasets can be filtered, mapped, and transformed 
        using built-in methods. Every dataset on the Hub has a dataset card documenting 
        its contents, intended use, and known limitations. The library supports automatic 
        train, validation, and test splits."""
    },
    {
        "id": "doc_5",
        "title": "What is the Inference API?",
        "text": """The Hugging Face Inference API allows you to run machine learning models 
        directly through HTTP requests without downloading them locally. It supports thousands 
        of models hosted on the Hub across tasks like text classification, text generation, 
        image classification, and more. The free tier allows limited requests per hour. 
        For production use, Inference Endpoints provide dedicated, scalable deployment. 
        The pipeline function in the transformers library can use the Inference API as a 
        backend by passing use_auth_token parameter. Inference API responses include the 
        model output and confidence scores where applicable."""
    },
    {
        "id": "doc_6",
        "title": "What is RAG — Retrieval Augmented Generation?",
        "text": """Retrieval Augmented Generation (RAG) is a technique that combines information 
        retrieval with text generation to produce more accurate and grounded responses. Instead 
        of relying solely on the knowledge stored in model parameters, RAG retrieves relevant 
        documents from an external knowledge base and provides them as context to the language 
        model. The process has two main stages: indexing, where documents are chunked and 
        embedded into a vector store, and retrieval, where a query is embedded and used to 
        find the most similar document chunks. RAG reduces hallucination, allows the model 
        to cite sources, and enables use of private or recent data not in the training set."""
    }
]

print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(f"  - {doc['title']}")

Loaded 6 documents
  - What is the Hugging Face Hub?
  - What are Transformers?
  - What is Fine-tuning?
  - What are Datasets in HuggingFace?
  - What is the Inference API?
  - What is RAG — Retrieval Augmented Generation?


In [4]:
# Cell 3 — Chunking
# For short docs like ours, each doc = one chunk
# For longer docs you'd split into overlapping pieces

def chunk_documents(documents, chunk_size=200, overlap=20):
    chunks = []
    for doc in documents:
        words = doc["text"].split()
        
        if len(words) <= chunk_size:
            # Short doc — keep as single chunk
            chunks.append({
                "chunk_id": f"{doc['id']}_chunk_0",
                "doc_id": doc["id"],
                "title": doc["title"],
                "text": doc["text"],
                "word_count": len(words)
            })
        else:
            # Long doc — split with overlap
            start = 0
            chunk_idx = 0
            while start < len(words):
                end = min(start + chunk_size, len(words))
                chunk_text = " ".join(words[start:end])
                chunks.append({
                    "chunk_id": f"{doc['id']}_chunk_{chunk_idx}",
                    "doc_id": doc["id"],
                    "title": doc["title"],
                    "text": chunk_text,
                    "word_count": len(words[start:end])
                })
                start += chunk_size - overlap
                chunk_idx += 1
    
    return chunks

chunks = chunk_documents(documents)

print(f"Documents: {len(documents)}")
print(f"Chunks:    {len(chunks)}")
print(f"\nSample chunk:")
print(f"  ID:    {chunks[0]['chunk_id']}")
print(f"  Title: {chunks[0]['title']}")
print(f"  Words: {chunks[0]['word_count']}")
print(f"  Text:  {chunks[0]['text'][:100]}...")

Documents: 6
Chunks:    6

Sample chunk:
  ID:    doc_1_chunk_0
  Title: What is the Hugging Face Hub?
  Words: 79
  Text:  The Hugging Face Hub is a platform with over 350,000 models, 75,000 datasets, 
        and 150,000 d...


In [5]:
# Cell 4 — Generate embeddings
from sentence_transformers import SentenceTransformer
import numpy as np

print("Loading embedding model...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded.")

# Extract just the text from each chunk
texts = [chunk["text"] for chunk in chunks]

# Generate embeddings for all chunks
print("\nGenerating embeddings...")
embeddings = embedder.encode(texts, show_progress_bar=True)

print(f"\nEmbedding matrix shape: {embeddings.shape}")
print(f"Each chunk → vector of {embeddings.shape[1]} numbers")
print(f"\nFirst 10 numbers of chunk 1 embedding:")
print(np.round(embeddings[0][:10], 4))

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded.

Generating embeddings...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix shape: (6, 384)
Each chunk → vector of 384 numbers

First 10 numbers of chunk 1 embedding:
[-0.0198 -0.0796  0.0169  0.0328  0.0856 -0.0314 -0.1085  0.0079  0.002
  0.0082]


In [6]:
# Cell 4b — Visualise semantic similarity
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

similarity_matrix = cosine_similarity(embeddings)

print("Semantic similarity between documents (0=different, 1=identical):\n")
titles_short = ["Hub", "Transformers", "Fine-tuning", "Datasets", "Inference API", "RAG"]

print(f"{'':15}", end="")
for t in titles_short:
    print(f"{t:14}", end="")
print()

for i, t1 in enumerate(titles_short):
    print(f"{t1:15}", end="")
    for j in range(len(titles_short)):
        val = similarity_matrix[i][j]
        print(f"{val:.2f}{'':10}", end="")
    print()

Semantic similarity between documents (0=different, 1=identical):

               Hub           Transformers  Fine-tuning   Datasets      Inference API RAG           
Hub            1.00          0.24          0.27          0.63          0.68          0.23          
Transformers   0.24          1.00          0.29          0.30          0.38          0.41          
Fine-tuning    0.27          0.29          1.00          0.48          0.36          0.40          
Datasets       0.63          0.30          0.48          1.00          0.62          0.34          
Inference API  0.68          0.38          0.36          0.62          1.00          0.29          
RAG            0.23          0.41          0.40          0.34          0.29          1.00          


In [7]:
# Cell 5 — Build FAISS vector index
import faiss
import numpy as np

# FAISS requires float32
embeddings_f32 = embeddings.astype(np.float32)

# Get embedding dimension
dimension = embeddings_f32.shape[1]  # 384

# Create a flat L2 index — simplest and most accurate
# For millions of vectors you'd use approximate indexes (IVF, HNSW)
index = faiss.IndexFlatL2(dimension)

# Add all embeddings to the index
index.add(embeddings_f32)

print(f"FAISS index built.")
print(f"Index type:      FlatL2")
print(f"Dimension:       {dimension}")
print(f"Vectors stored:  {index.ntotal}")
print(f"\nIndex is ready to search.")

FAISS index built.
Index type:      FlatL2
Dimension:       384
Vectors stored:  6

Index is ready to search.


In [8]:
# Cell 6 — Retrieval function
def retrieve(query, index, chunks, embedder, top_k=3):
    # Step 1 — Embed the query using same model
    query_embedding = embedder.encode([query]).astype(np.float32)
    
    # Step 2 — Search FAISS for top_k nearest vectors
    distances, indices = index.search(query_embedding, top_k)
    
    # Step 3 — Return the matching chunks with scores
    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0])):
        chunk = chunks[idx]
        results.append({
            "rank": rank + 1,
            "score": float(dist),
            "title": chunk["title"],
            "text": chunk["text"],
            "chunk_id": chunk["chunk_id"]
        })
    
    return results

# Test retrieval with a real question
query = "How do I use a model without downloading it?"
results = retrieve(query, index, chunks, embedder, top_k=3)

print(f"Query: '{query}'")
print(f"\nTop {len(results)} retrieved chunks:\n")
for r in results:
    print(f"Rank {r['rank']} — {r['title']}")
    print(f"  Distance score: {r['score']:.4f} (lower = more similar)")
    print(f"  Preview: {r['text'][:100]}...")
    print()

Query: 'How do I use a model without downloading it?'

Top 3 retrieved chunks:

Rank 1 — What is the Inference API?
  Distance score: 1.2619 (lower = more similar)
  Preview: The Hugging Face Inference API allows you to run machine learning models 
        directly through H...

Rank 2 — What is the Hugging Face Hub?
  Distance score: 1.3696 (lower = more similar)
  Preview: The Hugging Face Hub is a platform with over 350,000 models, 75,000 datasets, 
        and 150,000 d...

Rank 3 — What is Fine-tuning?
  Distance score: 1.5018 (lower = more similar)
  Preview: Fine-tuning is the process of taking a pretrained model and further training it 
        on a smalle...



In [9]:
# Cell 7 — Generation with Groq
from groq import Groq
import os

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

def generate_answer(query, retrieved_chunks, client):
    # Build context from retrieved chunks
    context = ""
    for chunk in retrieved_chunks:
        context += f"\n--- Source: {chunk['title']} ---\n"
        context += chunk["text"] + "\n"
    
    # Construct the RAG prompt
    prompt = f"""You are a helpful assistant that answers questions about HuggingFace.
Answer the question using ONLY the information provided in the context below.
If the answer is not in the context, say "I don't have enough information to answer that."
Always cite which source(s) you used.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.1,
        max_tokens=500
    )
    
    return response.choices[0].message.content

# Test end-to-end
query = "How do I use a model without downloading it?"
retrieved = retrieve(query, index, chunks, embedder, top_k=3)
answer = generate_answer(query, retrieved, client)

print(f"Q: {query}")
print(f"\nA: {answer}")

Q: How do I use a model without downloading it?

A: You can use a model without downloading it by utilizing the Hugging Face Inference API, which allows you to run machine learning models directly through HTTP requests (Source: What is the Inference API?). This way, you can access thousands of models hosted on the Hub without having to download them locally.


In [10]:
# Cell 8 — Test suite
test_questions = [
    "What is LoRA and how does it reduce parameters?",
    "What paper introduced the transformer architecture?",
    "How does HuggingFace store datasets efficiently?",
    "What is the price of HuggingFace Pro subscription?"  # not in docs
]

print("=" * 60)
for question in test_questions:
    retrieved = retrieve(question, index, chunks, embedder, top_k=2)
    answer = generate_answer(question, retrieved, client)
    
    print(f"\nQ: {question}")
    print(f"A: {answer}")
    print("-" * 60)


Q: What is LoRA and how does it reduce parameters?
A: LoRA (Low-Rank Adaptation) is a parameter-efficient method for fine-tuning pretrained models. It reduces trainable parameters by up to 99% while maintaining performance by adding low-rank matrices to existing weights, rather than updating all model parameters. (Source: "What is Fine-tuning?")
------------------------------------------------------------

Q: What paper introduced the transformer architecture?
A: The paper 'Attention is All You Need' introduced the transformer architecture in 2017. (Source: What are Transformers?)
------------------------------------------------------------

Q: How does HuggingFace store datasets efficiently?
A: HuggingFace stores datasets efficiently by using Apache Arrow as its backend for fast, memory-efficient data processing, and it also supports streaming for large datasets that don't fit in memory. (Source: What are Datasets in HuggingFace?)
-----------------------------------------------------

In [11]:
# Cell 9 — Gradio chat interface
import gradio as gr

def rag_chat(question, history):
    if not question.strip():
        return "Please ask a question."
    
    # Retrieve relevant chunks
    retrieved = retrieve(question, index, chunks, embedder, top_k=3)
    
    # Generate answer
    answer = generate_answer(question, retrieved, client)
    
    # Add sources used
    sources = "\n\n**Sources consulted:**\n"
    for r in retrieved:
        sources += f"- {r['title']} (distance: {r['score']:.3f})\n"
    
    return answer + sources

demo = gr.ChatInterface(
    fn=rag_chat,
    title="HuggingFace Knowledge Assistant",
    description="Ask questions about HuggingFace — answers grounded in documentation",
    examples=[
        "What is the HuggingFace Hub?",
        "How does fine-tuning work?",
        "What is RAG?",
        "How do transformers use attention?"
    ]
)

demo.launch(share=False)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [13]:
ls -lh rag.ipynb

-rw-r--r--@ 1 pragyansh  staff    26K May 27 16:40 rag.ipynb


In [14]:
git add rag.ipynb
git commit -m "Add complete notebook with all cell outputs"
git push


SyntaxError: invalid syntax (1897047401.py, line 1)